In [ ]:
'''
Waterfowl energetics model
Mike Mitchell - mmitchell@ducks.org - Ducks Unlimited
'''
name = 'default'
start_date = "Aug 1 2023"
numofdays = 228
removewater = False
customcurves = True
changecurvepct = False
smoothcrops = True
kcalperduck = 295
removeteal = False
geese = True
kcalpergoose = 500
removemoistsoil = False
lowmoistsoil = False
highmoistsoil = False
setcrops = False
setcropavailability = [100,100,100]
goosecurve = {
    'Arkansas_mav':{'pop':1238550,'curve':[0,0,.1,.5,1,1,.5,.3]},
    'Arkansas_wg':{'pop':0,'curve':[0,0,.1,.5,1,1,.5,.3]},
    'Illinois':{'pop':0,'curve':[0,0,.1,.5,1,1,.5,.3]},
    'Kentucky':{'pop':0,'curve':[0,0,.1,.3,1,1,.8,.2]},
    'Louisiana_mav':{'pop':164000,'curve':[0,0,0,.3,1,1,.5,.2]},
    'Louisiana_wg':{'pop':0,'curve':[0,0,0,.3,1,1,.5,.2]},
    'Mississippi':{'pop':270000,'curve':[0,0,0,.3,1,.3,.3,.2]},
    'Missouri':{'pop':0,'curve':[0,0,0,.3,1,.3,.3,.2]},
    'Oklahoma':{'pop':0,'curve':[0,0,.1,.5,1,1,.5,.3]},
    'Tennessee':{'pop':52000,'curve':[0,0,.1,.5,1,1,.5,.3]},
    'Texas':{'pop':0,'curve':[0,0,.1,.5,1,1,.5,.3]}
    }
goosereductionpct = 50
goosecrops = ['corn', 'milo', 'sorghum', 'rice','soybeans']
# Waterfowl in this analysis
keepducks = ['AGWT', 'AMWI', 'BWTE', 'GADW', 'MALL', 'NOPI', 'NSHO', 'RNDU', 'WODU']

baseaoiurl = 'https://giscog.blob.core.windows.net/waterfowlmodel/'
aois = ['ARmav', 'ARwg', 'KY', 'LAmav', 'LAwg', 'MO', 'MS', 'OK', 'TN', 'TX']

'''
# habitat type: 
    'energy (dud)':{
        Lo: unharvested dud, 
        Hi: harvested dud}, 
    'habitat availability': [curve with values between 0 and 100], 
    'decomposition': percentage per day  1% = 1, not 0.01
'''
cropdict = {
    'aquaculture': {
        'energy':{
            'Lo':3, 
            'High':3
            },
        'availability': [34,94,100], 
        'decomp':0
        },
    'corn': {
        'energy':{
            'Lo':1130,
            'High':27717
            },
        'availability':[3,100,27], 
        'decomp':1.57
        },
    'emergentwetlands': {
        'energy':{
            'Lo':283,
            'High':283
            },
        'availability':[19,89,100], 
        'decomp':0.18
        },
    'hardwoods': {
        'energy':{
            'Lo':23, 
            'High':917
            },
        'availability':[2,89,100], 
        'decomp':0.0037
        },
    'millet': {
        'energy':{
            'Lo':2638,
            'High':4338
            },
        'availability':[46,100,95], 
        'decomp':0.64
        },
    'milo': {
        'energy':{
            'Lo':1171,
            'High':8305
            },
        'availability':[3,100,30], 
        'decomp':0.322
        },     
    'moistsoil': {
        'energy':{
            'Lo':247, 
            'High':3513
            },
        'availability':[19,89,100], 
        'decomp':0.18
        },             
    'openwater': {
        'energy':{
            'Lo':3, 
            'High':3
            },
        'availability':[100,100,100], 
        'decomp':0.18
        },  
    'rice': {
        'energy':{
            'Lo':1099,
            'High':18602
            },
        'availability':[9,100,39], 
        'decomp':0.213
        },  
    'sorghum': {
        'energy':{
            'Lo':1171, 
            'High':8305
            },
        'availability':[3,100,30], 
        'decomp':0.322
        },  
    'soybeans': {
        'energy':{
            'Lo':248, 
            'High':5389
            },
        'availability':[5,100,38],
        'decomp':1.9
        }, 
    'woodywetlands': {
        'energy':{
            'Lo':23, 
            'High':917
            },
        'availability':[2,89,100], 
        'decomp':0.0037
        },              
    'wrp': {
        'energy':{
            'Lo':147, 
            'High':147
            },
        'availability':[2,89,100], 
        'decomp':0.0037
        }
}

# Habitat curve should be in percentages and max shouldn't be more than 100.
cropcurvedata = {key: val['availability'] for key, val in cropdict.items()}
# Decomposition rates as a daily percent decay.  To be calculated by day as leftover = leftover * (100-decay)
cropdecomp = {key: val['decomp'] for key, val in cropdict.items()}

customcurvesdict = {
                    'Arkansas_mav_MALL': [0, 0.0, 0.014301824, 0.017315462,0.38, 0.53, 1, 0.77, 0.52, 0.12],
                    'Arkansas_wg_MALL': [0, 0, 0.007728601,0.014762799,0.30,0.33,0.57,1.00,0.68,0.14,0],
                    #'Arkansas_mav_Other': [0,0.186634469,0.288705009,0.72791718,0.37037037,0.740740741,0.962962963,1,0.5,0.542658406],
                    #'Louisiana_wg_MALL':,
                    'Louisiana_mav_MALL': [0,0.04,0.52,0.80,1.00,0.51,0.20],
                    'Mississippi_MALL':[0, 0,0.027916566, 0.056929146, 0.448400933, 0.111111111, 0.555555556, 1, 0.888888889, 0.444444444, 0.120744769, 0],
                    'Tennessee_MALL':[0, 0, 0.2, 0.2 ,0.27,0.23,0.50,0.78,1.00,0.79,0.76,0.50,0.41],
                    #'Louisiana_mav_Other': [0,0.38,0.59,0.93,1.00,1.00,0.63,]
                   }

In [ ]:
##########################################
# All config above this line
##########################################

In [ ]:
# Checks to make sure habitat curves have at least 3 values and they are not > 100%        
for key,val in cropcurvedata.items():
    if max(val) >100:
        print('Value in {} has a value greater than 100'.format(key))
    if len(val) <3:
        print('Not a great curve with < 3 values for {}'.format(key))

if removewater:
    if 'openwater'in cropcurvedata:
        del cropcurvedata['openwater']

if removemoistsoil:
    if 'moistsoil'in cropcurvedata:
        del cropcurvedata['moistsoil']

if setcrops:
    for key in cropcurvedata:
        cropcurvedata[key] = setcropavailability

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import scipy
import sys
import pandas as pd
import json, requests
import geopandas as gpd
import shapely
import duckdb
import os
from shapely import wkt, wkb
import matplotlib.ticker as ticker
from datetime import datetime, timedelta
from shapely.geometry import Point
from scipy.spatial import cKDTree
from urllib.parse import urljoin
from scipy.interpolate import UnivariateSpline
import plotly.express as px
import plotly.io as pio

pd.set_option('display.float_format', lambda x : "{:,.2f}".format(x))
# Generate 228 dates starting from start_date defined above
start_date = datetime.strptime(start_date, "%b %d %Y")
date_labels = [(start_date + timedelta(days=i)).strftime("%b_%d")+' day: '+str(i+1) for i in range(numofdays)]

energycsvurl = urljoin(baseaoiurl + '/','4D_LMVJV-ST_Obj_FINAL_LONG2.csv')
bcrlocation = 'https://giscog.blob.core.windows.net/publicparquet/birdconservationregions_5070.parquet'
energydataset = '/mnt/e/source/waterfowl/waterfowlenergynotebook/energy_9_2025.parquet' # https://giscog.blob.core.windows.net/publicparquet/energy_9_2025.parquet

In [ ]:
# Setup duckdb
con = duckdb.connect()
con.install_extension("spatial")
con.load_extension("spatial")
con.install_extension("azure")
con.load_extension("azure")

In [ ]:
##########################################
#
# Setup done
#
##########################################

In [ ]:
# Flatten into a list of records
records = []
for crop, values in cropdict.items():
    record = {
        'Class': crop,
        'ValueLo': values['energy']['Lo'],
        'ValueHi': values['energy']['High'],
        'availability': values['availability'],
        'decomp': values['decomp']
    }
    records.append(record)

# Convert to DataFrame
df = pd.DataFrame(records)

In [ ]:
if lowmoistsoil:
    df.loc[df['Class'] == 'moistsoil', 'ValueHi'] = df.loc[df['Class'] == 'moistsoil', 'ValueLo']
if highmoistsoil:
    df.loc[df['Class'] == 'moistsoil', 'ValueLo'] = df.loc[df['Class'] == 'moistsoil', 'ValueHi']

In [ ]:
con.register('habitat', df)
display(df)

In [ ]:
# Read in energy dataset
con.sql('''
CREATE OR REPLACE TABLE lmvjvwaterfowlenergy AS SELECT * FROM read_parquet('{0}')
'''.format(energydataset))
con.sql('UPDATE lmvjvwaterfowlenergy SET Class = lower(Class)')
con.sql('''UPDATE lmvjvwaterfowlenergy SET CLASS = replace(CLASS, ' ', '')''')
con.sql('ALTER TABLE lmvjvwaterfowlenergy ADD COLUMN ValueLo DOUBLE;')
con.sql('ALTER TABLE lmvjvwaterfowlenergy ADD COLUMN ValueHi DOUBLE;')
con.sql('ALTER TABLE lmvjvwaterfowlenergy ADD COLUMN energyvalue DOUBLE;')
con.sql('''
UPDATE lmvjvwaterfowlenergy AS e
SET 
    ValueLo = c.ValueLo,
    ValueHi = c.ValueHi
FROM habitat AS c
WHERE e.Class = c.Class;
''')

In [ ]:
if removewater:
    con.sql('''DELETE FROM lmvjvwaterfowlenergy WHERE Class = 'openwater';''')

if removemoistsoil:
    con.sql('''DELETE FROM lmvjvwaterfowlenergy WHERE Class = 'moistsoil';''')

In [ ]:
con.sql('''
UPDATE lmvjvwaterfowlenergy
SET HrvstPCT = CASE
    WHEN source != 'wmu' THEN 100
    ELSE HrvstPCT
END;
''')
con.sql('''
UPDATE lmvjvwaterfowlenergy
SET OakPCT = CASE
    WHEN source NOT IN ('wmu','wrp') THEN 20
    ELSE OakPCT
END;
''')

In [ ]:
con.sql('''
UPDATE lmvjvwaterfowlenergy
SET energyvalue = CASE 
    WHEN CLASS = 'wrp' THEN ValueHi
    WHEN CLASS = 'moistsoil' THEN 
        CASE 
            WHEN lower(MSProd) = 'low' THEN ValueLo
            WHEN lower(MSProd) = 'high' THEN ValueHi
            WHEN lower(MSProd) = 'medium' THEN (ValueHi - ValueLo) / 2
            ELSE NULL
        END
        WHEN CLASS IN ('woodywetlands', 'hardwoods') THEN 
            CASE 
                WHEN OakPCT <= 0 THEN 0
                WHEN OakPCT < 10 THEN (OakPCT / 10.0) * ValueLo
                ELSE ((OakPCT - 10.0) / 90.0) * (ValueHi - ValueLo) + ValueLo
            END
    WHEN CLASS IN ('corn', 'rice', 'soybeans', 'milo', 'millet') THEN 
        HrvstPCT*.01 * ValueLo + (100 - HrvstPCT)*.01 * ValueHi    
    ELSE ValueLo
END;
''')


In [ ]:
tocrs = '5070'
aoiquery = 'https://services2.arcgis.com/5I7u4SJE1vUr79JC/ArcGIS/rest/services/LMVJV_Boundary/FeatureServer/0/query?where=1%3D1&objectIds=&time=&geometry=&geometryType=esriGeometryEnvelope&inSR=&spatialRel=esriSpatialRelIntersects&resultType=none&distance=0.0&units=esriSRUnit_Meter&relationParam=&returnGeodetic=false&outFields=&returnGeometry=true&returnCentroid=false&featureEncoding=esriDefault&multipatchOption=xyFootprint&maxAllowableOffset=&geometryPrecision=&outSR={0}&defaultSR=&datumTransformation=&applyVCSProjection=false&returnIdsOnly=false&returnUniqueIdsOnly=false&returnCountOnly=false&returnExtentOnly=false&returnQueryGeometry=false&returnDistinctValues=false&cacheHint=false&orderByFields=&groupByFieldsForStatistics=&outStatistics=&having=&resultOffset=&resultRecordCount=&returnZ=false&returnM=false&returnExceededLimitFeatures=true&quantizationParameters=&sqlFormat=none&f=pgeojson&token='.format(tocrs)
r = json.dumps(requests.get(aoiquery).json())
aoiresult = gpd.read_file(r)
aoibounds = list(aoiresult.bounds.values[0])

con.sql('''CREATE OR REPLACE TABLE cnty AS SELECT FIPS, NAME, STATE_NAME, geometry FROM read_parquet('https://giscog.blob.core.windows.net/abdu/uscounties.parquet') 
            WHERE ST_Intersects(geometry, ST_MakeEnvelope({0},{1},{2},{3}))'''.format(aoibounds[0],aoibounds[1],aoibounds[2],aoibounds[3]))

#inbcr = 'https://gisweb.ducks.org/server/rest/services/GEODATA/BCR/FeatureServer/0/query?where=1%3D1&objectIds=&time=&geometry=&geometryType=esriGeometryEnvelope&inSR=&spatialRel=esriSpatialRelIntersects&distance=&units=esriSRUnit_Foot&relationParam=&outFields=*&returnGeometry=true&maxAllowableOffset=&geometryPrecision=&outSR=5070&havingClause=&gdbVersion=&historicMoment=&returnDistinctValues=false&returnIdsOnly=false&returnCountOnly=false&returnExtentOnly=false&orderByFields=&groupByFieldsForStatistics=&outStatistics=&returnZ=false&returnM=false&multipatchOption=xyFootprint&resultOffset=&resultRecordCount=&returnTrueCurves=false&returnExceededLimitFeatures=false&quantizationParameters=&returnCentroid=false&timeReferenceUnknownClient=false&sqlFormat=none&resultType=&featureEncoding=esriDefault&datumTransformation=&f=geojson'
#r = json.dumps(requests.get(inbcr).json())
#bcr = gpd.read_file(r)
#bcr['geometry'] = bcr.to_wkb().geometry

In [ ]:
# Read in BCR
con.sql('''CREATE OR REPLACE TABLE bcr AS SELECT * FROM read_parquet('{0}')'''.format(bcrlocation))
con.sql('''CREATE OR REPLACE TABLE bcr AS SELECT BCR, BCRNAME, geometry FROM bcr''')

In [ ]:
con.sql('''CREATE OR REPLACE TABLE statebcr AS 
SELECT NAME, FIPS, STATE_NAME, BCR, BCRNAME, ST_Intersection(cnty.geometry, bcr.geometry) as geometry
FROM cnty
JOIN
bcr ON ST_Intersects(cnty.geometry, bcr.geometry)
''')

In [ ]:
con.sql(
    """
        CREATE OR REPLACE TABLE statebcrenergy AS 
        SELECT c.FIPS, c.NAME, c.STATE_NAME, c.BCR, c.BCRNAME, e.CLASS, e.source, e.energyvalue, ST_Intersection(e.geometry, c.geometry) as geometry
        FROM lmvjvwaterfowlenergy AS e, statebcr AS c
        WHERE ST_Intersects(e.geometry, c.geometry)
    """
)

In [ ]:
con.sql('''
UPDATE statebcrenergy
SET energyvalue = habitat.ValueLo
FROM habitat
WHERE statebcrenergy.Class = habitat.Class
  AND statebcrenergy.energyvalue IS NULL;
''')

In [ ]:
# Energy manipulation
# All energy comes in as DED per acre.  Convert to kcal
con.sql('''UPDATE statebcrenergy
SET energyvalue = energyvalue * {0}
'''.format(295))

In [ ]:
inenergyread = con.sql('select * exclude geometry, ST_AsText(geometry) as geometry from statebcrenergy').df()

In [ ]:
'''
Prep energy layer so we can calculate total energy at the county level.  Remove NAN
'''
inenergy = inenergyread.copy()
#print(inenergy['Class'].unique())
inenergy['geometry'] = inenergy['geometry'].apply(wkt.loads)
inenergy = gpd.GeoDataFrame(inenergy, geometry='geometry', crs=5070)
inenergy = inenergy.rename(columns={'Class':'CLASS'})
inenergy['acres'] = inenergy.area* 0.000247105
inenergy = inenergy.drop(['geometry'], axis=1)
#print(inenergy[['CLASS', 'energyvalue']].isna().groupby(inenergy['CLASS']).sum())

In [ ]:
inenergy.columns

In [ ]:
inenergy = inenergy.groupby(['STATE_NAME','FIPS', 'BCR','BCRNAME','CLASS','source','energyvalue']).agg({'acres': 'sum'}).reset_index()
inenergy = inenergy[inenergy['STATE_NAME']!='NaN']
#inenergy['statebcr'] = inenergy['statebcr'].astype('object')
inenergy['STATE_NAME'] = inenergy['STATE_NAME'].astype('object')
inenergy.loc[(inenergy['STATE_NAME']=='Arkansas') & (inenergy['BCRNAME']=='MISSISSIPPI ALLUVIAL VALLEY'), ['statebcr']] = 'Arkansas_mav'
inenergy.loc[(inenergy['STATE_NAME']=='Arkansas') & (inenergy['BCRNAME']=='WEST GULF COASTAL PLAIN/OUACHITAS'), ['statebcr']] = 'Arkansas_wg'
inenergy.loc[(inenergy['STATE_NAME']=='Louisiana') & (inenergy['BCRNAME']=='MISSISSIPPI ALLUVIAL VALLEY'), ['statebcr']] = 'Louisiana_mav'
inenergy.loc[(inenergy['STATE_NAME']=='Louisiana') & (inenergy['BCRNAME']=='WEST GULF COASTAL PLAIN/OUACHITAS'), ['statebcr']] = 'Louisiana_wg'
inenergy.loc[~inenergy['STATE_NAME'].isin(['Arkansas', 'Louisiana']), ['statebcr']] = inenergy['STATE_NAME']
inenergy = inenergy.drop(['STATE_NAME','BCR','BCRNAME'], axis=1)
print(inenergy['CLASS'].unique())

In [ ]:
print('Acres:', f"{int(inenergy['acres'].sum()):,}")
print(inenergy.groupby(['statebcr'])['acres'].sum())
inenergy['totalkcals'] = inenergy['acres']*inenergy['energyvalue']
print('')
print('kcals total:', f"{int(inenergy['totalkcals'].sum()):,}")
print(inenergy.groupby('statebcr')['totalkcals'].sum())
inenergy.groupby('statebcr')['totalkcals'].sum().to_csv(os.path.join('output','{0}_outputresults.csv'.format(name)))

In [ ]:
#pd.set_option('display.max_rows', None)   # Show all rows
display(inenergy.groupby(['CLASS'])['acres'].sum())

In [ ]:
# Read population objectives.  These are DUDs
popobjtable = pd.read_csv(energycsvurl)
popobjtable = popobjtable.rename(columns={'State':'state', 'LMVJV.80P.OBJ':'popobj80'})
print('Max population objective (DUD): {0:,.0f}'.format(popobjtable['popobj80'].max()))
print('Sum population objective (DUD): {0:,.0f}'.format(popobjtable['popobj80'].sum()))
popobjtable.groupby('state')['popobj80'].sum()

In [ ]:
# Read waterfowl curves and adjust attributes to align with long term objectives by statebcr
mergecurve = pd.DataFrame()
for aoi in aois:
    incsv = pd.read_csv(baseaoiurl+aoi+'daily_obj3.csv')
    mergecurve = pd.concat([mergecurve, incsv])
mergecurve = mergecurve.rename(columns={'ST':'state','SP2':'species'})
mergecurve.loc[mergecurve.state=='ARmav', ['state']] = 'Arkansas_mav'
mergecurve.loc[mergecurve.state=='ARwg', ['state']] = 'Arkansas_wg'
mergecurve.loc[mergecurve.state=='KY', ['state']] = 'Kentucky'
mergecurve.loc[mergecurve.state=='LAwg', ['state']] = 'Louisiana_wg'
mergecurve.loc[mergecurve.state=='LAmav', ['state']] = 'Louisiana_mav'
mergecurve.loc[mergecurve.state=='MO', ['state']] = 'Missouri'
mergecurve.loc[mergecurve.state=='MS', ['state']] = 'Mississippi'
mergecurve.loc[mergecurve.state=='OK', ['state']] = 'Oklahoma'
mergecurve.loc[mergecurve.state=='TN', ['state']] = 'Tennessee'
mergecurve.loc[mergecurve.state=='TX', ['state']] = 'Texas'

In [ ]:
mergecurve = mergecurve[mergecurve['species'].isin(keepducks)]
if removeteal:
    mergecurve = mergecurve[~mergecurve['species'].isin(['BWTE', 'AGWT'])]
mergecurve['species'].unique()

In [ ]:
# Merge population objectives and waterfowl curves.  Scale curves so the population objective is the max on the curve.
curvetable = pd.merge(mergecurve, popobjtable, on=('state', 'species'), how='left')
curvetable['max'] = curvetable.select_dtypes(include=[np.float64]).drop(columns=['popobj80']).max(axis=1)
curvetable = curvetable[curvetable['popobj80']>0]
curvetable['scale'] = curvetable['popobj80']/curvetable['max']
newtable = curvetable[curvetable.select_dtypes(include=['float64']).columns].multiply(curvetable['scale'],axis='index')
newtable = newtable.drop(columns=['popobj80', 'scale'])
curvetable.update(newtable)
curvetable = curvetable.drop(['Unnamed: 0'], axis=1)
curvetable = curvetable.drop(columns=['scale', 'popobj80'])
curvetable = curvetable.rename(columns={'state':'statebcr'})

In [ ]:
current_date = datetime.now()
formatted_date = current_date.strftime("%m_%d_%Y")

#curvetable.to_parquet(os.path.join('output','popcurve_{0}_{1}.parquet'.format(formatted_date, name)))

In [ ]:
# Read in decomp
indecomp = pd.DataFrame.from_dict(cropdecomp,orient='index', columns=['decomp'])
indecomp = indecomp.reset_index()
indecomp = indecomp.rename(columns={'index':'CLASS'})
inenergy = inenergy.merge(indecomp, on='CLASS', how='left')

In [ ]:
if not changecurvepct:
    if customcurves:
        for k, v in customcurvesdict.items():
            sp = k.rsplit("_",1)[1]
            st = k.rsplit("_",1)[0]
            if sp == 'Other':
                continue
            forreplace = np.interp(np.linspace(0, numofdays, numofdays), [round(p) for p in np.linspace(0, numofdays, len(v))], v)
            print(sp)
            print(st)
            spmax = curvetable[(curvetable['species']==sp)&(curvetable['statebcr']==st)]['max'].iloc[0]
            newcurve = forreplace*spmax

            # Interpolate to 229 points
            x_interp = np.linspace(0, numofdays, numofdays)
            x_known = [round(p) for p in np.linspace(0, numofdays, len(v))]
            v_interp = forreplace

            # Smooth using UnivariateSpline
            spline = UnivariateSpline(x_interp, v_interp)
            spline.set_smoothing_factor(0.01)
            v_smooth = spline(x_interp)

            # Scale smoothed values to preserve the original max
            original_max = np.max(v_interp)
            smoothed_max = np.max(v_smooth)
            v_smooth = v_smooth * (original_max / smoothed_max) *spmax
            v_smooth = v_smooth.clip(min=0)

            plt.plot(date_labels,list(curvetable[(curvetable['species']==sp)&(curvetable['statebcr']==st)].drop(columns=['species', 'statebcr']).T[:-1].T.iloc[0]), label='Original', alpha=0.5)
            plt.plot(date_labels, v_smooth, label='Replaced', linewidth=2)
            plt.legend()
            plt.title("Curve replacement for {0} in {1}".format(sp, st))
            plt.xticks(np.arange(0, 230, step=20), rotation='vertical')
            plt.xlabel("Date")
            plt.ylabel("# of birds")
            plt.savefig('./output/plots/CurveChange_{2}_{0}_{1}.png'.format(sp, st, name), bbox_inches='tight')
            plt.show()
            # Replace the values
            curvetable.loc[(curvetable['species'] == sp) & (curvetable['statebcr'] == st), [str(i) for i in range(1, numofdays+1)]] = v_smooth        

In [ ]:
if changecurvepct:
    for st, sp in zip(curvetable['statebcr'], curvetable['species']):
        spmax = curvetable[(curvetable['species']==sp)&(curvetable['statebcr']==st)]['max'].iloc[0]
        newcurve = [changecurvepct*spmax] * numofdays
        curvetable.loc[(curvetable['species'] == sp) & (curvetable['statebcr'] == st), [str(i) for i in range(1, numofdays+1)]] = newcurve

In [ ]:
# Plot by species
import math
curvetable = curvetable.replace([np.inf, -np.inf], np.nan).fillna(0)
curveforplot = curvetable.copy()
colors=['red', 'black', 'blue', 'brown', 'green', 'pink', 'cyan', 'purple', 'orange', 'yellow', 'grey', 'lime']
sq = int(math.sqrt(len(curvetable['statebcr'].unique()))+1)
a=0
c=0
d=0
for stbcr in curveforplot['statebcr'].unique():
    i=0
    for sp in curveforplot['species'].unique():
        #print(sp, i)
        ct = curveforplot[curveforplot['statebcr']==stbcr]
        tmp = ct.drop(columns='max').groupby('species').sum().drop(columns='statebcr')
        maxsp = ct[['statebcr', 'max', 'species']].groupby('species').sum().drop(columns='statebcr')
        maxsp = maxsp.reset_index()
        maxsp = maxsp[maxsp['species']==sp]
        tmp = tmp.reset_index()
        tmp = tmp[tmp['species']==sp]
        tmp = tmp.drop(['species'], axis=1).transpose().reset_index()
        try:
            plt.plot(date_labels,tmp[i], color=colors[i], label=sp)
            plt.axhline(y = maxsp['max'][i], color=colors[i], linestyle = '--', label='_') 
        except:
            continue
        i+=1
    plt.legend(framealpha=1, bbox_to_anchor=(1.25, 1),loc='upper right')
    plt.ticklabel_format(style='plain', axis='y')
    plt.xticks(np.arange(0, 230, step=20), rotation='vertical')
    plt.title('Species population curves with max line for {0}'.format(stbcr.replace('_', ' ')), pad=15)
    plt.xlabel('Day', labelpad=10)
    plt.ylabel('Population objective (DUD)', labelpad=10)
    plt.yticks(rotation=45)
    plt.gca().yaxis.set_major_formatter(ticker.StrMethodFormatter('{x:,.0f}'))
    plt.savefig('./output/plots/{1}_{0}.png'.format(stbcr, name), bbox_inches='tight')
    plt.show()

In [ ]:
curvetable = curvetable.groupby(['statebcr']).sum().reset_index()#.drop(['Unnamed: 0'], axis=1)
curvetable = curvetable.drop(columns='species')
curvetable = curvetable.replace([np.inf, -np.inf], np.nan).fillna(0)
curvetable = curvetable.drop(columns=['max'])

In [ ]:
# sum energy demand by statebcr, removing species.
#display(curvetable.head())
i=0
aucsp = {}
for st in curvetable['statebcr'].unique():
    query = pd.DataFrame(curvetable[curvetable['statebcr']==st]).drop('statebcr',axis=1).transpose().reset_index()
    plt.plot(date_labels, query[i])
    aucsp[st]=(np.trapezoid(query[i].apply(int), query['index'].apply(int)))
    i+=1
plt.legend(curvetable['statebcr'].unique())    
plt.title('Energy demand by state / bcr')
plt.xticks(np.arange(0, 230, step=20), rotation='vertical')
plt.xlabel('Day', labelpad=10)
plt.ylabel('# of birds')
plt.yticks(rotation=45)
plt.gca().yaxis.set_major_formatter(ticker.StrMethodFormatter('{x:,.0f}'))
plt.savefig('./output/plots/{1}_{0}.png'.format('demandbystatebcr', name), bbox_inches='tight')

In [ ]:
############ PREP Goose data ############
goosecurvekcal = pd.DataFrame()

# Create column names as strings "1" to "228"

rows=[]

if geese:
    for k, v in goosecurve.items():
        curve = goosecurve[k]['curve']
        spmax = goosecurve[k]['pop'] * goosereductionpct*0.01
        if spmax == 0:
            continue
        # Interpolate to 229 points
        x_interp = np.linspace(0, numofdays, numofdays)
        x_known = [round(p) for p in np.linspace(0, numofdays, len(v))]
        v_interp = np.interp(np.linspace(0, numofdays, numofdays), [round(p) for p in np.linspace(0, numofdays, len(curve))], curve)

        # Smooth using UnivariateSpline
        spline = UnivariateSpline(x_interp, v_interp)
        spline.set_smoothing_factor(0.01)
        v_smooth = spline(x_interp)

        # Scale smoothed values to preserve the original max
        original_max = np.max(v_interp)
        smoothed_max = np.max(v_smooth)
        v_smooth = v_smooth * (original_max / smoothed_max) * spmax
        v_smooth = v_smooth.clip(min=0)

        # Create DataFrame with one row (statebcr = 'k') and 228 columns
        row = {'statebcr': k}
        row.update({str(i + 1): v for i, v in enumerate(v_smooth)})
        rows.append(row)
        
        plt.plot(date_labels, v_smooth, label='_Replaced', linewidth=2)
        plt.title("Goose curve for {0}".format(k))
        plt.xticks(np.arange(0, 230, step=20), rotation='vertical')
        plt.xlabel("Date")
        plt.ylabel("# of birds")
        plt.show()            
    goosecurvekcal = pd.DataFrame(rows)

In [ ]:
display(curvetable)
totaldemand = curvetable.sum().reset_index().drop([0])
display(totaldemand)
plt.plot(date_labels, totaldemand[0]*kcalperduck)
plt.xticks(np.arange(0, 230, step=20), rotation='vertical')
plt.title('Total Demand')
plt.xlabel('Day', labelpad=10)
plt.ylabel('Energy (kcal)')
plt.yticks(rotation=45)
plt.gca().yaxis.set_major_formatter(ticker.StrMethodFormatter('{x:,.0f}'))
plt.savefig('./output/plots/{1}_{0}.png'.format('TotalDemand',name), bbox_inches='tight')

In [ ]:
# Calculate habitat curves
pio.renderers.default = 'notebook'  # or 'notebook_connected', or 'iframe_connected'

habitatcurvedata = {}

for sp in cropcurvedata.keys():
    # Interpolation to daily steps
    x_interp = np.linspace(1, numofdays, numofdays)
    x_known = [round(p) for p in np.linspace(1, numofdays, len(cropcurvedata[sp]))]
    v_interp = np.interp(x_interp, x_known, cropcurvedata[sp])

    if smoothcrops:
        spline = UnivariateSpline(x_interp, v_interp)
        spline.set_smoothing_factor(1000)
        v_smooth = spline(x_interp)
        v_smooth = v_smooth * (100.0 / np.max(v_smooth))
        v_smooth = v_smooth.clip(min=0)
        habitatcurvedata[sp] = v_smooth
    else:
        habitatcurvedata[sp] = v_interp

# Convert to DataFrame (wide format)
habitatcurve = pd.DataFrame.from_dict(habitatcurvedata).transpose().reset_index().rename(columns={'index':'CLASS'})
habitatcurve.columns = ['CLASS'] + list(range(1, numofdays + 1))

# Convert to long format for Plotly
habitatcurve_long = habitatcurve.melt(id_vars='CLASS', var_name='Day', value_name='PercentHabitat')

# Optional: Add a real date label column if needed
# habitatcurve_long['Date'] = pd.to_datetime('2024-07-01') + pd.to_timedelta(habitatcurve_long['Day'] - 1, unit='D')

# Plotly interactive line plot
fig = px.line(
    habitatcurve_long,
    x='Day',
    y='PercentHabitat',
    color='CLASS',
    title='Habitat availability over time',
    labels={
        'Day': 'Day',
        'PercentHabitat': '% habitat available',
        'CLASS': 'Class'
    },
    hover_name='CLASS',
    hover_data={'PercentHabitat': ':.2f'}
)

fig.update_layout(
    xaxis=dict(tickmode='linear', tick0=0, dtick=20),
    yaxis=dict(tickformat=".0f"),
    height=600
)
fig.write_html(f'./output/plots/{name}_habitatenergycurve.html')
fig.show()


In [ ]:
#inenergy.to_parquet('energy_{0}.parquet'.format(formatted_date))

In [ ]:
inenergy = inenergy.dropna(subset=['statebcr'])
print('Total kcal supply:', f"{int(inenergy['totalkcals'].sum()):,}")
print(inenergy.groupby(['CLASS'])['totalkcals'].sum().reset_index())
print(inenergy[inenergy['statebcr']=='Louisiana_mav'].groupby(['statebcr','CLASS'])['totalkcals'].sum().reset_index())

In [ ]:
inenergy = inenergy.dropna(subset=['source'])
print('Total kcal supply:', f"{int(inenergy['totalkcals'].sum()):,}")
print(inenergy.groupby(['source','CLASS'])['totalkcals'].sum().reset_index())

In [ ]:
# Energy under the curve
tmpx = totaldemand['index'].apply(int)
tmpy = totaldemand[0].apply(int)
totaldemandarea = np.trapezoid(tmpy,tmpx)
print('Demand (DUD) AuC:',f'{totaldemandarea:,.0f}')
print('Demand (DUD) sum:', f'{tmpy.sum():,.0f}')
print('Demand (kcal) sum:', f'{tmpy.sum()* kcalperduck:,.0f}')
print('Energy supply (kcal) sum:',f'{round(inenergy.totalkcals).sum():,.0f}')
print('Energy supply (DUD) sum:',f'{round(inenergy.totalkcals/kcalperduck).sum():,.0f}')
print('############')
print('Demand (DUD)')
print(pd.DataFrame({'statebcr': list(aucsp.keys()), 'demand': list(aucsp.values())}))
blah = inenergy.groupby(['statebcr', 'totalkcals'],as_index=False).sum()[['statebcr', 'totalkcals', 'acres']]
blah['energy supply'] = blah['totalkcals']/kcalperduck
print('Energy supply (DUD)')
print(blah.groupby('statebcr', as_index=False).sum()[['statebcr', 'energy supply']])

fordiffdemand=pd.DataFrame({'statebcr': list(aucsp.keys()), 'demand': list(aucsp.values())})
fordiffsupply=blah.groupby('statebcr', as_index=False).sum()[['statebcr', 'energy supply']]
mergediff = fordiffsupply.merge(fordiffdemand, on='statebcr')
mergediff['diff'] = mergediff['energy supply'] - mergediff['demand']
print(mergediff)

In [ ]:
################ Start daily energy calculation ################

In [ ]:
# testing daily iteration and aggregation
trackenergy = pd.DataFrame() # create empty dataframe to hold output by day
inenergy.replace([np.inf, -np.inf], np.nan).fillna(0)
energylayer = inenergy.copy().fillna(0)
energylayer['unique'] = energylayer.index
energylayer['vegenergyprevLo'] = 0
energylayer['leftoverLo'] = 1
energylayer['startacres'] = energylayer['acres'] 
energylayer['reduceacres'] = 0
energylayer['totalsupplyLo'] = energylayer['acres'] * energylayer['energyvalue']
trackhideficit = pd.DataFrame()
tracklodeficit = pd.DataFrame()
pd.set_option('display.max_columns', None)

for i in range(1,numofdays+1): #1 to numofdays
    energylayer['day'] = i
    # Get habitat availability based on habitat curve and calculate available acres
    hab = habitatcurve[['CLASS', i]] # select habitat availability curve for day by class
    energylayer = energylayer.merge(hab, on='CLASS', how='left') # merge habitat curve to the energy layer
    energylayer['habpct'] = energylayer[i]
    energylayer = energylayer.drop(i, axis=1) # Drop habitat percentage day column
    energylayer['acres'] = (energylayer['acres'] - energylayer['reduceacres']).clip(lower=0)
    energylayer['availacres'] = energylayer['acres'] * energylayer['habpct']*.01# calculate available acres which is acres of the energy polygon * habitat type availability for that day.
    
    # Calculate energy supply
    energylayer['vegenergyLo'] = energylayer['availacres'] * energylayer['energyvalue']   
    energylayer['reserveEnergyLo'] = energylayer['totalsupplyLo'] - energylayer['vegenergyLo']
    energylayer['reserveAcres'] = energylayer['acres'] - energylayer['availacres']
    energylayer['reservekcal'] = energylayer['reserveAcres'] * energylayer['energyvalue'] 
    energylayer['diffLo'] = (energylayer['vegenergyLo'] - energylayer['vegenergyprevLo']).clip(lower=0) # Energy supply includes the leftover energy from the day before plus the difference between todays supply energy and yesterdays.
    energylayer['vegenergyprevLo'] = energylayer['vegenergyLo']
    # Add supply from previous day
    #energylayer['supplyLo'] = energylayer['vegenergyLo']
    energylayer['supplyLo'] = energylayer['leftoverLo']  + energylayer['diffLo']
    
    # Proportion demand based on energy supply at the record level.
    filtersupplyLo = energylayer[energylayer['supplyLo'] >= 0]
    energylayerbystatebcr = filtersupplyLo[['statebcr','supplyLo']].groupby(['statebcr']).sum().rename(columns={'supplyLo':'supplyLoMax'})
    if 'supplyLoMax' in energylayer.columns:
        energylayer = energylayer.drop('supplyLoMax', axis=1)
    energylayer = energylayer.merge(energylayerbystatebcr, on='statebcr', how='left')
    #energylayer['pctdemand'] = energylayer['supplyLo']/energylayer['supplyLoMax']
    def calculate_pctdemand(group):
        if (group['leftoverLo'] < 0).any():
            total_supply = group['totalsupplyLo'].sum()
            group['pctdemand'] = group['totalsupplyLo'] / total_supply if total_supply != 0 else 0
        else:
            group['pctdemand'] = group['supplyLo'] / group['supplyLoMax']
        return group

    # Apply logic per statebcr
    

    energylayer = (
        energylayer
        .groupby("statebcr", group_keys=False)
        .apply(lambda g: calculate_pctdemand(g).assign(statebcr=g.name), include_groups=False)
    )

    energylayer.loc[np.isnan(energylayer['pctdemand']),['pctdemand']] = 0
    popcurve = curvetable[['statebcr', str(i)]] # select demand for the day based on curve.
    
    energylayer = energylayer.merge(popcurve, on='statebcr', how='left') # merge demand for that day based on statebcr
    energylayer['demand'] = abs(energylayer['pctdemand']*energylayer[str(i)]*kcalperduck)
    energylayer = energylayer.drop(str(i), axis=1, errors='ignore')
    # Calculate leftover energy
    energylayer['leftoverLo'] = energylayer['supplyLo'] - energylayer['demand']
    
    # Calculate reduction in acres from goose forage
    if geese:
        goosepopcurve = goosecurvekcal[['statebcr', str(i)]]
        energylayer = energylayer.merge(goosepopcurve, on='statebcr', how='left') # merge demand for that day based on statebcr
        energylayer['goosedemandkcal'] = 0
        energylayer[str(i)] = energylayer[str(i)].fillna(0)
        energylayer[str(i)] = energylayer[str(i)].astype('int64')
        energylayer.loc[energylayer['CLASS'].isin(goosecrops), 'goosedemandkcal'] = energylayer[str(i)] * kcalpergoose
        energylayer['reduceacres'] = (energylayer['goosedemandkcal'] / energylayer['energyvalue']).clip(lower=0)
        energylayer = energylayer.drop(str(i), axis=1, errors='ignore')
    
    # Decomp  *** need to only factor in decomp is leftover is positive.
    energylayer.loc[energylayer['leftoverLo'] > 0, 'leftoverLo'] *= ((100 - energylayer['decomp']) * 0.01)

    trackenergy = pd.concat([trackenergy, energylayer])    
    tracklodeficit = pd.concat([tracklodeficit,energylayer[['day','statebcr','leftoverLo']].groupby(['day','statebcr']).sum().reset_index()])

In [ ]:
print('Day 1')
sample = trackenergy[trackenergy['day']==1]
printme = sample[['CLASS', 'startacres','acres', 'availacres', 'reduceacres','supplyLo', 'totalsupplyLo', 'reserveEnergyLo','demand', 'leftoverLo']].groupby(['CLASS']).sum()
with pd.option_context('display.max_rows', None, 'display.max_columns', None):  # more options can be specified also
    display(printme)
printme = printme.reset_index()
print('Day 2')
sample = trackenergy[trackenergy['day']==2]
printme = sample[['CLASS', 'startacres','acres', 'availacres','reduceacres','supplyLo', 'totalsupplyLo', 'reserveEnergyLo','demand', 'leftoverLo']].groupby(['CLASS']).sum()
with pd.option_context('display.max_rows', None, 'display.max_columns', None):  # more options can be specified also
    display(printme)
printme = printme.reset_index()

In [ ]:
#sample = trackenergy[trackenergy['day']==1]
sample = trackenergy[(trackenergy['day']==3) & (trackenergy['statebcr']=='Texas')]
printme = sample[['statebcr','CLASS', 'startacres','acres', 'availacres','reduceacres','supplyLo', 'totalsupplyLo', 'reserveEnergyLo','demand', 'leftoverLo']].groupby(['statebcr','CLASS']).sum()
print(printme.sum())
with pd.option_context('display.max_rows', None, 'display.max_columns', None):  # more options can be specified also
    display(printme)
printme = printme.reset_index()

In [ ]:
sample = trackenergy[trackenergy['day']==numofdays]
printme = sample[['statebcr', 'startacres', 'acres', 'availacres','supplyLo', 'totalsupplyLo', 'reserveEnergyLo','demand', 'leftoverLo']].groupby(['statebcr']).sum()
print('Values in kcal')
print(printme.sum())
with pd.option_context('display.max_rows', None, 'display.max_columns', None):  # more options can be specified also
    display(printme)
print('\nThese ran out of energy')
with pd.option_context('display.max_rows', None, 'display.max_columns', None):  # more options can be specified also
    display(printme[printme['leftoverLo']<0])    
printme = printme.reset_index()

In [ ]:
sample = trackenergy[trackenergy['day']==numofdays]
printme = sample[['statebcr', 'leftoverLo']].groupby(['statebcr']).sum()
printme['leftoverLo'] = printme['leftoverLo']/kcalperduck
print('Values in DED')
print(printme.sum())
with pd.option_context('display.max_rows', None, 'display.max_columns', None):  # more options can be specified also
    display(printme)
printme.to_csv(os.path.join('output','{0}_outputresults.csv'.format(name)))    
print('\nThese ran out of energy')
with pd.option_context('display.max_rows', None, 'display.max_columns', None):  # more options can be specified also
    display(printme[printme['leftoverLo']<0])    
printme = printme.reset_index()

In [ ]:
'''
plt.plot(list(range(1,229)),trackhideficit)
plt.plot(list(range(1,229)),tracklodeficit)
plt.title('Energy deficit over time')
plt.legend(['High', 'Low'])
'''

tracklodeficit = tracklodeficit[tracklodeficit['statebcr'] != 0]
#tracklodeficit = tracklodeficit[tracklodeficit['statebcr'] != 'Kentucky']
for st in tracklodeficit['statebcr'].unique():
    query = tracklodeficit[tracklodeficit['statebcr']==st].drop(['statebcr'],axis=1) 
    plt.plot(date_labels, query[['leftoverLo']])
plt.legend(tracklodeficit['statebcr'].unique(), ncol=2)#,bbox_to_anchor=(1.35, 1),loc='upper right')
#plt.ticklabel_format(style='plain')
plt.title('Leftover low by statebcr')
plt.xlabel('Day', labelpad=10)
plt.xticks(np.arange(0, 230, step=20), rotation='vertical')
plt.ylabel('kcal')
plt.yticks(rotation=45)
plt.gca().yaxis.set_major_formatter(ticker.StrMethodFormatter('{x:,.0f}'))
plt.savefig('./plots/{0}.png'.format('LeftoverLowbyState'), bbox_inches='tight')

In [ ]:
for stbcr in tracklodeficit['statebcr'].unique():
    query = tracklodeficit[tracklodeficit['statebcr']==stbcr].drop(['statebcr'],axis=1)
    plt.plot(query[['day']], query[['leftoverLo']])
    plt.ticklabel_format(style='plain')
    plt.title('Leftover energy low for {0}'.format(stbcr))
    plt.xlabel('Day')
    plt.ylabel('kcal')
    plt.yticks(rotation=45)
    plt.gca().yaxis.set_major_formatter(ticker.StrMethodFormatter('{x:,.0f}'))
    plt.savefig('./plots/leftoverenergy_{0}.png'.format(stbcr), bbox_inches='tight')
    plt.show()

In [ ]:
import plotly.express as px
import plotly.io as pio
pio.renderers.default = 'notebook'
# Plot all habtiat types leftoverlo
agg_df = (
    trackenergy
    .groupby(['statebcr', 'CLASS', 'day'], as_index=False)
    .agg({'leftoverLo': 'sum'})
)

# Loop through each statebcr and generate an interactive plot
for stbcr in agg_df['statebcr'].unique():
    query = agg_df[agg_df['statebcr'] == stbcr]

    fig = px.line(
        query,
        x='day',
        y='leftoverLo',
        color='CLASS',
        title=f'Leftover energy low for {stbcr}',
        labels={
            'leftoverLo': 'kcal',
            'day': 'Day',
            'CLASS': 'Class'
        },
        hover_name='CLASS',
        hover_data={'leftoverLo': ':.0f', 'day': True}
    )
    
    fig.update_layout(
        yaxis_tickformat=',',
        yaxis_title='kcal',
        xaxis_title='Day',
        legend_title='Class',
        height=600
    )
    fig.write_html(f'./output/plots/{name}_{stbcr}_leftoverenergyinteractivee.html')
    fig.show()


agg_df = (
    trackenergy
    .groupby(['statebcr', 'CLASS', 'day'], as_index=False)
    .agg({'leftoverLo': 'sum'})
)

for stbcr in agg_df['statebcr'].unique():
    query = agg_df[agg_df['statebcr'] == stbcr]

    plt.figure(figsize=(10, 6))

    for cls, subdf in query.groupby('CLASS'):
        subdf = subdf.sort_values('day')
        plt.plot(subdf['day'], subdf['leftoverLo'], label=str(cls))

    plt.ticklabel_format(style='plain')
    plt.title(f'Leftover energy low for {stbcr}')
    plt.xlabel('Day')
    plt.ylabel('kcal')
    plt.yticks(rotation=45)
    plt.legend(title='Class')
    plt.gca().yaxis.set_major_formatter(ticker.StrMethodFormatter('{x:,.0f}'))
    plt.savefig(f'./plots/leftoverenergyByClass_{name}_{stbcr}.png', bbox_inches='tight')
    #plt.show()
 

In [ ]:
#plt.plot(list(range(1,229)),trackenergy[['day','leftoverHi']].groupby(['day']).sum().reset_index()['leftoverHi'])
plt.plot(list(range(1,229)),trackenergy[['day','leftoverLo']].groupby(['day']).sum().reset_index()['leftoverLo'])
plt.title('Leftover low energy over time')
#plt.legend(['Low energy value'])
plt.xticks(np.arange(0, 230, step=20), rotation='vertical')
plt.xlabel('Day', labelpad=10)
plt.ylabel('Energy (kcal)')
plt.yticks(rotation=45)
plt.gca().yaxis.set_major_formatter(ticker.StrMethodFormatter('{x:,.0f}'))
plt.savefig('./plots/{0}.png'.format('LeftoverLowEnergyOverTime'), bbox_inches='tight')

In [ ]:
plt.plot(list(range(1,229)),trackenergy[['day','demand']].groupby(['day']).sum().reset_index()['demand'])
plt.title('Energy demand over time')
plt.xticks(np.arange(0, 230, step=20), rotation='vertical')
plt.xlabel('Day', labelpad=10)
plt.ylabel('Energy (kcal)')
plt.yticks(rotation=45)
plt.gca().yaxis.set_major_formatter(ticker.StrMethodFormatter('{x:,.0f}'))
plt.savefig('./plots/{0}.png'.format('DemandOverTime'), bbox_inches='tight')

In [ ]:
plt.plot(list(range(1,229)),trackenergy[['day','demand']].groupby(['day']).sum().reset_index()['demand'])
plt.plot(list(range(1,229)),trackenergy[['day','leftoverLo']].groupby(['day']).sum().reset_index()['leftoverLo'])
plt.legend(['Energy demand', 'Energy supply'])
plt.title('Log Energy supply and demand over time')
plt.yscale('log')
plt.xticks(np.arange(0, 230, step=20), rotation='vertical')
plt.xlabel('Day', labelpad=10)
plt.ylabel('Log energy (kcal)')
#plt.yticks(rotation=45)
#plt.gca().yaxis.set_major_formatter(ticker.StrMethodFormatter('{x:,.0f}'))
plt.savefig('./plots/{0}.png'.format('LogEnergyDemandAndSupply'), bbox_inches='tight')

In [ ]:
county = con.sql('''SELECT * from cnty''').df()
geoenergy = trackenergy.merge(county[['FIPS', 'geometry']], on=['FIPS'], how='left')
print(len(geoenergy.index))
#geoenergy.to_parquet('trackenergygeo.parquet')

In [ ]:
trackenergy.to_csv(os.path.join('output','trackenergy_{0}.csv'.format(name)))

In [ ]:
trackenergy.groupby('statebcr')['supplyLoMax'].min()